# Визуализация GPS-трека на карте

Чтение CSV с координатами, отрисовка точек с градиентом (синий → красный) и направление по полю **bearing**.

In [76]:
import os
import pandas as pd
import numpy as np
import folium
from matplotlib.colors import Normalize, to_hex
from matplotlib import cm

# Папка с датасетами
DATA_DIR = r"datasets/pre_datasets/Novosibirsk"
# DATA_DIR = r"datasets/row_datasets/Novosibirsk"

# Список CSV в папке
csv_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith('.csv')]
csv_files.sort()
print("Доступные файлы:", csv_files)

Доступные файлы: ['20260123_52_str_preprocessed.csv', '20260126_52_str_preprocessed.csv', '20260126_72_rev_preprocessed.csv', '20260203_52_rev_preprocessed.csv', '20260203_52_str_preprocessed.csv', '20260204_38_rev_preprocessed.csv', '20260204_38_str_preprocessed.csv', '20260206_38_rev_preprocessed.csv', '20260206_52_str_preprocessed.csv', '20260209_52_rev_preprocessed.csv', '20260209_52_str_preprocessed.csv', '20260210_52_rev_preprocessed.csv', '20260210_52_str_preprocessed.csv']


In [77]:
# Выберите файл: подставьте имя из списка выше или оставьте первый
selected_file = "20260123_52_str_preprocessed.csv"  # или любой другой из списка выш
# selected_file = "20260123_52_str.csv"
filepath = os.path.join(DATA_DIR, selected_file)

df = pd.read_csv(filepath)
df = df.dropna(subset=['lat', 'lon'])  # убираем строки без координат
print(f"Загружено строк: {len(df)}")
df.head()

Загружено строк: 1648


,time,accuracy,speed,route_frac,lat,lon,bearing,trip_id,vehicle_type,route_direction,segment_id,next_stop_index,eta_sec
0,2026-01-23 05:56:29.438000+00:00,4.331222,2.580000,0.664843,54.888855,83.075793,243.862731,20260123_52_str,bus,52_str,43,44,217.0
1,2026-01-23 05:56:30.438000+00:00,4.355726,3.415659,0.664951,54.888827,83.075732,237.756552,20260123_52_str,bus,52_str,43,44,216.0
2,2026-01-23 05:56:31.438000+00:00,4.380230,4.251318,0.665059,54.888800,83.075671,237.756552,20260123_52_str,bus,52_str,43,44,215.0
3,2026-01-23 05:56:32.438000+00:00,4.404733,5.086977,0.665167,54.888772,83.075611,237.756552,20260123_52_str,bus,52_str,43,44,214.0
4,2026-01-23 05:56:33.438000+00:00,4.429237,5.922636,0.665275,54.888745,83.075550,237.756552,20260123_52_str,bus,52_str,43,44,213.0


In [78]:
# Градиент синий → красный по порядку точек (индекс 0 = синий, последняя = красный)
norm = Normalize(vmin=0, vmax=max(1, len(df) - 1))
from matplotlib.colors import LinearSegmentedColormap

cmap = LinearSegmentedColormap.from_list(
    "blue_black_red",
    ["blue", "black", "red"]
)

def index_to_color(i):
    return to_hex(cmap(norm(i)))

# Длина "носика" направления в метрах
BEARING_LENGTH_M = 25

def bearing_to_endpoint(lat, lon, bearing_deg):
    """Точка впереди по направлению bearing (0=север, 90=восток), в метрах."""
    rad = np.deg2rad(bearing_deg)
    # приближение: 1° широты ≈ 111320 м, долгота с учётом cos(lat)
    m_per_deg_lat = 111320
    m_per_deg_lon = 111320 * np.cos(np.deg2rad(lat))
    d_north = BEARING_LENGTH_M * np.cos(rad)
    d_east = BEARING_LENGTH_M * np.sin(rad)
    lat2 = lat + d_north / m_per_deg_lat
    lon2 = lon + d_east / m_per_deg_lon
    return lat2, lon2

In [79]:
# Центр карты по средним координатам
center_lat = df['lat'].mean()
center_lon = df['lon'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='OpenStreetMap')

# Рисуем точки и "носики" направления
for idx, (_, row) in enumerate(df.iterrows()):
    lat, lon = row['lat'], row['lon']
    color = index_to_color(idx)
    folium.CircleMarker(
        location=[lat, lon],
        radius=7,
        color=color,
        fill=True,
        fill_color=color,
        weight=1,
    ).add_to(m)

    # Носик направления (bearing): только если значение задано
    b = row.get('bearing')
    if pd.notna(b) and b != '':
        try:
            bearing = float(b)
            lat2, lon2 = bearing_to_endpoint(lat, lon, bearing)
            folium.PolyLine(
                locations=[[lat, lon], [lat2, lon2]],
                color=color,
                weight=3,
                opacity=1,
            ).add_to(m)
        except (TypeError, ValueError):
            pass

# Сохраняем и открываем в браузере. Чтобы карта была видна и в ноутбуке: File → Trust Notebook, затем перезапустите ячейку
import webbrowser
map_path = os.path.abspath(os.path.join(os.path.dirname(DATA_DIR), 'map_track_output.html'))
m.save(map_path)
webbrowser.open('file:///' + map_path.replace(chr(92), '/'))
print('Карта открыта в браузере:', map_path)
m

Карта открыта в браузере: d:\!NGU!\4_year_2026\VKR\datasets\pre_datasets\map_track_output.html


Цвет точек: **синий** — начало трека, **красный** — конец. Короткий отрезок от точки показывает направление движения (**bearing**, градусы от севера по часовой).